**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 35: 프로젝트 성능 평가 및 반복 개선

## 🎯 모델 평가 및 개선

이 노트북에서는 파인튜닝된 모델의 성능을 **다각도로 평가**하고, **반복 개선 전략**을 수립합니다.

### 평가 프레임워크
```
평가 데이터 준비 → 자동 평가 → LLM-as-a-Judge → 정성 평가 → 개선 전략 → 반복 학습
```

### 평가 방법론
| 방법 | 측정 대상 | 자동화 | 신뢰도 |
|------|----------|--------|--------|
| **자동 메트릭** (ROUGE, BLEU) | 텍스트 유사도 | ✅ 완전 자동 | ⭐⭐ |
| **LLM-as-a-Judge** | 종합 품질 | ✅ 반자동 | ⭐⭐⭐⭐ |
| **정성 평가** | 실제 사용성 | ❌ 수동 | ⭐⭐⭐⭐⭐ |

### 사전 준비
- `34_project_training.ipynb`에서 저장한 파인튜닝 모델
- `eval.json` 평가 데이터
- OpenAI API 키 (LLM-as-a-Judge 사용 시)

In [ ]:
# =====================================================
# 📦 라이브러리 임포트
# =====================================================
import torch
import gc
import os
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import Dataset

print("✅ 라이브러리 임포트 완료")

def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")


In [ ]:
# =====================================================
# 🤖 모델 + LoRA 어댑터 로드 (FP16)
# =====================================================
OUTPUT_DIR = "./output/project"

with open(os.path.join(OUTPUT_DIR, "project_plan.json"), "r", encoding="utf-8") as f:
    project_plan = json.load(f)

MODEL_NAME = project_plan["model_config"]["base_model"]
ADAPTER_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")

print(f"⏳ 모델 로딩: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print(f"✅ 모델 + 어댑터 로드 완료")
print_gpu_memory("모델 로드 후")


---
## 1️⃣ 평가 데이터셋 준비

도메인에 맞는 테스트 셋을 준비합니다.

In [ ]:
# =====================================================
# 📊 평가 프롬프트로 모델 응답 생성
# =====================================================
EVAL_PROMPTS = project_plan["eval_prompts"]

model.eval()
predictions = []

print(f"📊 파인튜닝 모델 응답 생성")
print("="*60)

for prompt in EVAL_PROMPTS:
    messages = [
        {"role": "system", "content": "당신은 AI/ML 분야의 전문가입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    predictions.append(response)
    print(f"\n🔹 {prompt}")
    print(f"   {response[:200]}")

responses = predictions
print(f"\n✅ 응답 생성 완료: {len(predictions)}건")


In [ ]:
# =====================================================
# 📊 응답 품질 분석 (자동 메트릭)
# =====================================================
print("📊 응답 품질 분석")
print("="*60)

for i, (prompt, response) in enumerate(zip(EVAL_PROMPTS, responses)):
    tokens = tokenizer.encode(response)
    print(f"\n🔹 질문 {i+1}: {prompt[:40]}...")
    print(f"   응답 길이: {len(response)}자 / {len(tokens)} 토큰")
    print(f"   한국어 비율: {sum(1 for c in response if '가' <= c <= '힣') / max(len(response), 1) * 100:.0f}%")

avg_len = sum(len(r) for r in responses) / len(responses)
avg_tokens = sum(len(tokenizer.encode(r)) for r in responses) / len(responses)
print(f"\n📊 평균:")
print(f"   응답 길이: {avg_len:.0f}자 / {avg_tokens:.0f} 토큰")


---
## 2️⃣ LLM-as-a-Judge 평가

GPT-4를 활용하여 모델 출력의 품질을 **다차원으로 평가**합니다.

### 평가 차원
- **정확성 (Accuracy)**: 정보가 정확한가?
- **유용성 (Helpfulness)**: 답변이 도움이 되는가?
- **관련성 (Relevance)**: 질문에 적절히 답변하는가?
- **완결성 (Completeness)**: 충분히 상세한가?
- **안전성 (Safety)**: 유해하거나 부적절한 내용이 없는가?

In [ ]:
# =====================================================
# 🤖 LLM-as-a-Judge 평가 (OpenAI 또는 Ollama)
# =====================================================
import json, requests

# OpenAI 또는 Ollama 자동 선택
USE_OPENAI = False
try:
    from openai import OpenAI
    from dotenv import load_dotenv
    load_dotenv("../.env")
    client = OpenAI()
    client.models.list()
    USE_OPENAI = True
    print("✅ OpenAI API 사용 (gpt-4o-mini)")
except:
    print("⚠️ OpenAI 불가 → Ollama 사용")

def llm_judge(prompt, response):
    """LLM으로 응답 품질을 5가지 차원에서 1-5점으로 평가"""
    judge_prompt = f"""당신은 AI 응답 품질을 평가하는 전문 평가자입니다.

## 질문
{prompt}

## 응답
{response}

## 평가 기준 (각 1-5점)
1. 정확성 (Accuracy): 정보가 사실적이고 올바른가?
2. 유용성 (Helpfulness): 답변이 실질적으로 도움이 되는가?
3. 관련성 (Relevance): 질문에 적절히 답변하는가?
4. 완결성 (Completeness): 충분히 상세하고 빠진 내용이 없는가?
5. 명확성 (Clarity): 이해하기 쉽고 잘 구조화되어 있는가?

반드시 아래 JSON 형식으로만 답변하세요:
{{"accuracy": 점수, "helpfulness": 점수, "relevance": 점수, "completeness": 점수, "clarity": 점수, "reason": "한 줄 평가 이유"}}"""

    try:
        if USE_OPENAI:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": judge_prompt}],
                temperature=0.1,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        else:
            resp = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": "qwen2.5:1.5b", "prompt": judge_prompt, "stream": False},
                timeout=120,
            )
            text = resp.json().get("response", "")
            start = text.find("{")
            end = text.rfind("}") + 1
            if start >= 0 and end > start:
                return json.loads(text[start:end])
            return {"accuracy": 3, "helpfulness": 3, "relevance": 3, "completeness": 3, "clarity": 3, "reason": "파싱 실패"}
    except Exception as e:
        print(f"  ⚠️ 평가 실패: {e}")
        return {"accuracy": 3, "helpfulness": 3, "relevance": 3, "completeness": 3, "clarity": 3, "reason": "평가 실패"}

# 전체 평가 실행
print("🤖 LLM-as-a-Judge 평가 시작")
print("="*60)

judge_results = []
for i, (prompt, response) in enumerate(zip(EVAL_PROMPTS, predictions)):
    print(f"\n⏳ [{i+1}/{len(EVAL_PROMPTS)}] {prompt[:40]}...")
    score = llm_judge(prompt, response)
    judge_results.append(score)
    
    avg = (score["accuracy"] + score["helpfulness"] + score["relevance"] + score["completeness"] + score["clarity"]) / 5
    print(f"   정확성: {score['accuracy']} | 유용성: {score['helpfulness']} | 관련성: {score['relevance']} | 완결성: {score['completeness']} | 명확성: {score['clarity']} | 평균: {avg:.1f}")
    print(f"   사유: {score.get('reason', '')}")

# 종합 결과
print(f"\n{'='*60}")
print("📊 LLM-as-a-Judge 종합 결과")
print("="*60)

dimensions = ["accuracy", "helpfulness", "relevance", "completeness", "clarity"]
for dim in dimensions:
    scores = [r[dim] for r in judge_results]
    print(f"   {dim:15s}: {sum(scores)/len(scores):.1f} / 5.0")

all_scores = [sum(r[d] for d in dimensions)/5 for r in judge_results]
print(f"\n   {'종합 평균':15s}: {sum(all_scores)/len(all_scores):.1f} / 5.0")


---
## 3️⃣ 개선 방향 제안

RTX 4060 환경에서 적용 가능한 개선 전략입니다.


In [ ]:
# =====================================================
# 🔄 개선 방향 제안 (RTX 4060, Qwen2.5-1.5B + FP16 + LoRA 환경 기준)
# =====================================================
print("🔄 개선 방향 제안")
print("="*60)
print("   1. 학습 데이터 확장: 50건 → 200~500건 이상")
print("   2. 에포크 조정: 현재 loss 추이를 보고 증감 (오버피팅 주의)")
print("   3. 도메인 특화 데이터 추가 (GPT-4 / gpt-4o-mini 합성 생성)")
print("   4. seed_topics 다양화 및 품질 검수 강화")
print("   5. DPO로 응답 선호도 추가 개선 (preference 데이터 필요)")
print("   6. LoRA r 조정 (r=16 → r=32, OOM 여부 확인)")


---
## 4️⃣ 평가 결과 저장

평가 결과를 JSON 파일로 저장합니다.


In [ ]:
# =====================================================
# 💾 평가 결과 저장
# =====================================================
eval_save = {
    "eval_prompts": EVAL_PROMPTS,
    "responses": responses,
    "judge_results": judge_results,
}

eval_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_save, f, ensure_ascii=False, indent=2)
print(f"✅ 평가 결과 저장: {eval_path}")


---
## 5️⃣ GPU 메모리 정리 및 마무리

평가 모델을 메모리에서 해제합니다.

> 💡 **반복 학습**: 평가 결과가 만족스럽지 않다면 `34_project_training.ipynb`로 돌아가서 데이터/하이퍼파라미터를 조정하고 재학습 후 이 노트북을 다시 실행하세요.


In [ ]:
# =====================================================
# 🧹 GPU 메모리 정리
# =====================================================
del model, base_model
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("정리 후")
print("\n🎉 평가 완료! 다음 단계: 36번 노트북 (배포)")


---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 평가 데이터셋 기반 모델 응답 생성
- ✅ 응답 품질 기초 분석 (길이, 토큰 수, 한국어 비율)
- ✅ LLM-as-a-Judge 다차원 평가 (OpenAI / Ollama 자동 선택)
- ✅ 개선 방향 제안
- ✅ 평가 결과 저장

### 산출물
- 📁 `output/project/eval_results.json` - 평가 결과 (응답 + Judge 점수)

### 다음 노트북
👉 **36_project_deployment.ipynb**: 프로젝트 배포 및 어플리케이션 구현
